# 04 - Gold Business Model

Create analytics-ready dimensions, a sales fact table, and an aggregated KPI table.


In [ ]:
from pyspark.sql import functions as F

c=spark.table("silver_customers")
p=spark.table("silver_products")
s=spark.table("silver_sales")

fact=(s.withColumn("gross_profit",F.col("sales_amount")-F.col("cost_amount"))
      .withColumn("order_year",F.year("order_date"))
      .withColumn("order_month",F.month("order_date")))

summary=(fact
    .join(c.select("customer_id","segment","city","state","region"),"customer_id","left")
    .join(p.select("product_id","product_name","category"),"product_id","left")
    .groupBy("order_date","region","segment","category")
    .agg(F.sum("sales_amount").alias("total_sales"),
         F.sum("gross_profit").alias("gross_profit"),
         F.sum("quantity").alias("units_sold"),
         F.countDistinct("order_id").alias("orders")))

c.write.format("delta").mode("overwrite").option("overwriteSchema","true").saveAsTable("gold_dim_customer")
p.write.format("delta").mode("overwrite").option("overwriteSchema","true").saveAsTable("gold_dim_product")
fact.write.format("delta").mode("overwrite").option("overwriteSchema","true").saveAsTable("gold_fact_sales")
summary.write.format("delta").mode("overwrite").option("overwriteSchema","true").saveAsTable("gold_sales_summary")
print("Gold layer created.")
